In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("../dataset.csv")
df.head()

,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
0,63,1,True,True,True,Shahran,1.850000e+09,61666.67
1,60,1,True,True,True,Shahran,1.850000e+09,61666.67
2,79,2,True,True,True,Pardis,5.500000e+08,18333.33
3,95,2,True,True,True,Shahrake Qods,9.025000e+08,30083.33
4,123,2,True,True,True,Shahrake Gharb,7.000000e+09,233333.33


In [3]:
df.shape

(3479, 8)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3479 entries, 0 to 3478
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Area        3479 non-null   str    
 1   Room        3479 non-null   int64  
 2   Parking     3479 non-null   bool   
 3   Warehouse   3479 non-null   bool   
 4   Elevator    3479 non-null   bool   
 5   Address     3456 non-null   str    
 6   Price       3479 non-null   float64
 7   Price(USD)  3479 non-null   float64
dtypes: bool(3), float64(2), int64(1), str(2)
memory usage: 191.2 KB


In [5]:
df.isnull().sum()

Area           0
Room           0
Parking        0
Warehouse      0
Elevator       0
Address       23
Price          0
Price(USD)     0
dtype: int64

In [6]:
df["Address"].unique()

<ArrowStringArray>
[                   'Shahran',                     'Pardis',
              'Shahrake Qods',             'Shahrake Gharb',
 'North Program Organization',                   'Andisheh',
     'West Ferdows Boulevard',                     'Narmak',
                'Saadat Abad',                      'Zafar',
 ...
          'Thirteen November',                    'Darakeh',
              'Aliabad South',             'Alborz Complex',
                 'Firoozkooh',                  'Vahidiyeh',
                   'Shadabad',                   'Naziabad',
                  'Javadiyeh',                'Yakhchiabad']
Length: 193, dtype: str

In [7]:
df = df.drop("Price",axis = 1)

In [8]:
df = df.rename(columns ={"Price(USD)":"Price"} )

In [9]:
df = df.dropna()

In [10]:
df.isnull().sum()

Area         0
Room         0
Parking      0
Warehouse    0
Elevator     0
Address      0
Price        0
dtype: int64

In [11]:
df["Area"].unique()

<ArrowStringArray>
[             '63',              '60',              '79',              '95',
             '123',              '70',              '87',              '59',
              '54',              '71',
 ...
 ' 2,550,000,000 ',             '330',             '212',             '345',
             '530',             '630',              '30',             '205',
             '285',             '186']
Length: 243, dtype: str

In [12]:
df["Area"]=df["Area"].str.replace(",","")

In [13]:
df["Area"]=df["Area"].astype(int)

In [14]:
df["Price"]=round(df['Price'],0)

In [15]:
df["Price"]=df['Price'].astype(int)

In [16]:
df.head()

,Area,Room,Parking,Warehouse,Elevator,Address,Price
0,63,1,True,True,True,Shahran,61667
1,60,1,True,True,True,Shahran,61667
2,79,2,True,True,True,Pardis,18333
3,95,2,True,True,True,Shahrake Qods,30083
4,123,2,True,True,True,Shahrake Gharb,233333


In [17]:
clean_df = df

In [18]:
clean_df.describe()

,Area,Room,Price
count,3.456000e+03,3456.000000,3.456000e+03
mean,8.802191e+06,2.081308,1.793319e+05
std,3.177783e+08,0.759723,2.707243e+05
min,3.000000e+01,0.000000,1.200000e+02
25%,6.900000e+01,2.000000,4.733300e+04
50%,9.000000e+01,2.000000,9.666700e+04
75%,1.210000e+02,2.000000,2.000000e+05
max,1.616000e+10,5.000000,3.080000e+06


In [19]:
x = clean_df.drop("Price",axis=1)
y=clean_df[["Price"]]

In [20]:
x.isnull().sum()
y.isnull().sum()

Price    0
dtype: int64

Training and testing data

In [21]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)


Encoding categorical variables

Binary Mapping

In [22]:
x_train["Parking"]=x_train["Parking"].astype(int)
x_train["Elevator"]=x_train["Elevator"].astype(int)
x_train["Warehouse"]=x_train["Warehouse"].astype(int)
x_test["Parking"]=x_test["Parking"].astype(int)
x_test["Elevator"]=x_test["Elevator"].astype(int)
x_test["Warehouse"]=x_test["Warehouse"].astype(int)

One Hot Encoding

In [23]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(handle_unknown="ignore",sparse_output=False)

In [24]:
encoded_train= encoder.fit_transform(x_train[["Address"]])
encoded_test = encoder.transform(x_test[["Address"]])
encoded_train_df = pd.DataFrame(
    encoded_train,
    columns=encoder.get_feature_names_out(["Address"]),
    index = x_train.index
)
encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=encoder.get_feature_names_out(["Address"]),
    index = x_test.index
)

In [25]:
x_train = x_train.drop("Address",axis=1)
x_train = pd.concat([x_train,encoded_train_df],axis=1)
x_test = x_test.drop("Address",axis=1)
x_test = pd.concat([x_test,encoded_test_df],axis=1)

['../models/encoder.pkl']

Feature Scaling

In [27]:
from sklearn.preprocessing import StandardScaler

In [36]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled= scaler.transform(x_test)


['../models/scaler.pkl']

Training Regression Model

In [37]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [38]:
lr_model = LinearRegression()
dt_model = DecisionTreeRegressor()
rf_model = RandomForestRegressor()

In [39]:
models={
    "LinearRegression":lr_model,
    "DecisionTreeRegressor":dt_model,
    "RandomForestRegressor":rf_model
}
            

In [43]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
best_r2 = float("-inf")
best_model = None
for name,model in models.items():
    if name == "LinearRegression":
        model.fit(x_train_scaled, y_train)
        y_pred=model.predict(x_test_scaled)
    else:
        model.fit(x_train,y_train)
        y_pred=model.predict(x_test)
    print(name)
    print(("MAE:",round(mean_absolute_error(y_test,y_pred),3)))
    print("MSE:",round(mean_squared_error(y_test,y_pred),3))
    print("R2 score:",round(r2_score(y_test,y_pred),3))
    model_r2 = round(r2_score(y_test,y_pred),3)
    if model_r2 > best_r2:
        best_r2 = model_r2
        best_model = model
print(f"The best model is {best_model},with an r2 score of {best_r2} ")




LinearRegression
('MAE:', 96241.211)
MSE: 47084078446.871
R2 score: 0.471
DecisionTreeRegressor
('MAE:', 61971.793)
MSE: 33138592328.189
R2 score: 0.628


c:\Users\SM Computer\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor
('MAE:', 55200.963)
MSE: 22665247792.648
R2 score: 0.745
The best model is RandomForestRegressor(),with an r2 score of 0.745 


['../models/best_model.pkl']